# C2 - StudentLife DAILY FEATURE TABLE (step 2)

**Purpose:** turn the raw StudentLife files into ONE table with one row per **participant per day**.
This is feature building only. **No labels are invented and no model is trained here.**

## What this notebook builds
| Group | Columns | Built from | Meaning |
|---|---|---|---|
| Screen / phone use | `lock_minutes`, `n_lock_events` | `sensing/phonelock` | minutes the phone was locked, and how many lock periods started that day |
| Darkness | `dark_minutes` | `sensing/dark` | minutes the phone's sensor reported dark |
| Charging | `charge_minutes` | `sensing/phonecharge` | minutes the phone was charging |
| Sleep PROXY | `sleep_proxy_hours` | dark AND locked overlap | longest stretch (hours) that was both dark and locked and ended in the morning (04:00-14:00). **A heuristic guess, not measured sleep.** |
| Physical activity | `act_frac_<code>`, `act_n_samples` | `sensing/activity` | share of that day's activity samples per activity code |
| App usage | `app_n_snapshots`, `app_n_distinct_packages` | `app_usage` | how many app snapshots and how many different apps were seen (no app names are kept) |
| Reference (NOT inputs) | `stress_n_level_1..5`, `sleep_hours_reported`, `sleep_rate_code`, `sleep_social_code` | EMA Stress / Sleep | what participants said that day. Used later only to check proxies or as candidate targets |

## Assumptions (please read - these are my decisions, not facts from the files)
1. **Time zone:** timestamps are Unix seconds (UTC). The EMA GPS points (about 43.7, -72.3) are in Hanover, New Hampshire, so I convert to `America/New_York` to decide which calendar day an event belongs to. Dartmouth's study period includes a daylight-saving change, which the conversion handles.
2. **Sleep proxy** is only a heuristic (dark AND locked, ending between 04:00 and 14:00). It is preliminary and will be compared with the self-reported sleep below. It is not a medical or validated measure.
3. **Activity codes:** I keep the codes as they are (`act_frac_0`, `act_frac_1`, ...). Confirm what each code means in the StudentLife paper PDF in your folder before interpreting them.
4. **Stress `level` is NOT one ordered scale** (1 = a little stressed, 2 = definitely stressed, 3 = stressed out, 4 = feeling good, 5 = feeling great). So I only COUNT answers per level and never average them.
5. **Leakage warning:** daily features include the whole day, even events AFTER an EMA answer was given. So this daily table is for exploration only. Any real prediction must later use only data from BEFORE the answer time.

StudentLife is NOT mixed with SWELL-KW.

## Cell 1 - Connect Drive and find the dataset
Same as notebook 1. Click **Connect to Google Drive** if the permission box appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json
import numpy as np
import pandas as pd

def find_data_root(start='/content/drive/MyDrive', max_depth=10):
    """Search Drive for the folder that contains sensing/phonelock (the StudentLife root)."""
    start_depth = start.rstrip('/').count('/')
    for dirpath, dirnames, _ in os.walk(start):
        if dirpath.count('/') - start_depth >= max_depth:
            dirnames[:] = []
            continue
        if os.path.basename(dirpath) == 'sensing' and 'phonelock' in dirnames:
            return Path(dirpath).parent
    return None

DATA_ROOT = find_data_root()
assert DATA_ROOT is not None, 'StudentLife folder not found - set DATA_ROOT by hand.'
sensing_dir = DATA_ROOT / 'sensing'
ema_resp = DATA_ROOT / 'EMA' / 'response'

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)

## Cell 2 - Helper functions
Small reusable tools:
- `uid_from_name` gets the participant id (like `u07`) from a file name.
- `to_local` converts Unix seconds to the New York calendar day.
- `interval_minutes_per_day` splits each (start, end) interval across midnights and adds up minutes per day.
- `intersect_intervals` finds the time when two interval lists overlap (used for the sleep proxy).

In [ ]:
TZ = 'America/New_York'          # assumption 1 in the notebook header
MAX_INTERVAL_DAYS = 7            # ignore intervals longer than this (almost surely bad timestamps)

def uid_from_name(name):
    m = re.search(r'(u\d{2})', name)
    return m.group(1) if m else None

def files_by_uid(folder, ext):
    """Return {uid: [file paths]} for every file with the given extension."""
    out = {}
    for f in sorted(Path(folder).glob('*' + ext)):
        uid = uid_from_name(f.name)
        if uid:
            out.setdefault(uid, []).append(f)
    return out

def to_local(seconds):
    """Unix seconds (Series) -> timezone-aware local datetimes."""
    return pd.to_datetime(seconds, unit='s', utc=True).dt.tz_convert(TZ)

def read_intervals(paths):
    """Read one or more start/end files for a participant. Drops bad rows and reports how many."""
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df.columns = df.columns.str.strip()
    df = df[['start', 'end']].dropna().drop_duplicates()   # u20 has a duplicate file 'phonecharge_u20 (1).csv' - identical rows must not count twice
    good = (df['end'] > df['start']) & ((df['end'] - df['start']) <= MAX_INTERVAL_DAYS * 86400)
    n_bad = int((~good).sum())
    return df[good].sort_values('start').reset_index(drop=True), n_bad

def interval_minutes_per_day(df):
    """Total minutes per local calendar day, splitting intervals that cross midnight."""
    out = {}
    for s, e in zip(df['start'].values, df['end'].values):
        t = pd.Timestamp(s, unit='s', tz='UTC').tz_convert(TZ)
        t_end = pd.Timestamp(e, unit='s', tz='UTC').tz_convert(TZ)
        while t < t_end:
            next_midnight = t.normalize() + pd.DateOffset(days=1)
            seg_end = min(t_end, next_midnight)
            d = t.date()
            out[d] = out.get(d, 0.0) + (seg_end - t).total_seconds() / 60.0
            t = seg_end
    return pd.Series(out, dtype=float)

def intersect_intervals(a, b):
    """Overlap of two sorted interval tables (each with start, end)."""
    A, B = a[['start', 'end']].values, b[['start', 'end']].values
    i = j = 0
    res = []
    while i < len(A) and j < len(B):
        s, e = max(A[i][0], B[j][0]), min(A[i][1], B[j][1])
        if e > s:
            res.append((s, e))
        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1
    return pd.DataFrame(res, columns=['start', 'end'])

def sleep_proxy_hours(inter):
    """Per local day: longest overlap (hours) of dark+locked that ENDED between 04:00 and 14:00."""
    if inter.empty:
        return pd.Series(dtype=float)
    end_local = to_local(inter['end'])
    hours = (inter['end'] - inter['start']) / 3600.0
    morning = (end_local.dt.hour >= 4) & (end_local.dt.hour < 14)
    d = pd.DataFrame({'date': end_local.dt.date[morning], 'h': hours[morning]})
    return d.groupby('date')['h'].max()
print('helpers ready')

## Cell 3 - Phone lock, dark, charging and the sleep proxy (per participant, per day)
Loops over every participant. For each one it reads the three interval files and builds the daily columns.
Anything it cannot read is printed (never silently skipped).

In [ ]:
lock_files   = files_by_uid(sensing_dir / 'phonelock', '.csv')
dark_files   = files_by_uid(sensing_dir / 'dark', '.csv')
charge_files = files_by_uid(sensing_dir / 'phonecharge', '.csv')
all_uids = sorted(set(lock_files) | set(dark_files) | set(charge_files))

for name, d in [('phonelock', lock_files), ('dark', dark_files), ('phonecharge', charge_files)]:
    multi = {u: [p.name for p in ps] for u, ps in d.items() if len(ps) > 1}
    if multi:
        print(f'NOTE {name}: participants with more than one file -> {multi}')

interval_rows = []
bad_report = []
for uid in all_uids:
    cols = {}
    lock = dark = None
    if uid in lock_files:
        lock, nb = read_intervals(lock_files[uid]); bad_report.append((uid, 'phonelock', nb))
        cols['lock_minutes'] = interval_minutes_per_day(lock)
        starts_local = to_local(lock['start']).dt.date
        cols['n_lock_events'] = starts_local.value_counts().astype(float)
    if uid in dark_files:
        dark, nb = read_intervals(dark_files[uid]); bad_report.append((uid, 'dark', nb))
        cols['dark_minutes'] = interval_minutes_per_day(dark)
    if uid in charge_files:
        ch, nb = read_intervals(charge_files[uid]); bad_report.append((uid, 'phonecharge', nb))
        cols['charge_minutes'] = interval_minutes_per_day(ch)
    if lock is not None and dark is not None:
        cols['sleep_proxy_hours'] = sleep_proxy_hours(intersect_intervals(lock, dark))
    df_u = pd.DataFrame(cols)
    df_u.index.name = 'date'
    df_u['uid'] = uid
    interval_rows.append(df_u.reset_index())

interval_feats = pd.concat(interval_rows, ignore_index=True)
print('interval features:', interval_feats.shape, '| participants:', interval_feats['uid'].nunique())
n_bad_total = sum(n for _, _, n in bad_report)
print('interval rows dropped as invalid (end<=start or longer than', MAX_INTERVAL_DAYS, 'days):', n_bad_total)
display(interval_feats.head())

## Cell 4 - Physical activity (share of samples per activity code, per day)
`activity` files are large (hundreds of thousands of rows each), so this cell is the slowest. Note the real column name has a **leading space**
(` activity inference`), which is why columns are stripped.

In [ ]:
def activity_daily(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df['date'] = to_local(df['timestamp']).dt.date
    counts = df.groupby(['date', 'activity inference']).size().unstack(fill_value=0)
    frac = counts.div(counts.sum(axis=1), axis=0)
    frac.columns = [f'act_frac_{c}' for c in frac.columns]
    frac['act_n_samples'] = counts.sum(axis=1)
    return frac

act_files = files_by_uid(sensing_dir / 'activity', '.csv')
act_rows = []
for uid, paths in act_files.items():
    try:
        a = pd.concat([activity_daily(p) for p in paths])
        a = a.groupby(level=0).mean()          # only matters if a participant had >1 file
        a['uid'] = uid
        act_rows.append(a.reset_index())
    except Exception as e:
        print('Could not process activity for', uid, '->', e)

activity_feats = pd.concat(act_rows, ignore_index=True)
print('activity features:', activity_feats.shape, '| participants:', activity_feats['uid'].nunique())
print('activity columns:', [c for c in activity_feats.columns if c.startswith('act_')])
display(activity_feats.head())

## Cell 5 - App usage (counts only, no app names kept)
Reads just two columns of each big `app_usage` file (`timestamp` and the top-activity package name) to save memory.
We keep only **how many** snapshots and **how many different** apps per day. App names are privacy-sensitive and are dropped immediately.

In [ ]:
APP_PKG_COL = 'RUNNING_TASKS_topActivity_mPackage'

def app_daily(path):
    df = pd.read_csv(path, usecols=['timestamp', APP_PKG_COL])
    df['date'] = to_local(df['timestamp']).dt.date
    g = df.groupby('date')
    return pd.DataFrame({'app_n_snapshots': g['timestamp'].nunique(),
                         'app_n_distinct_packages': g[APP_PKG_COL].nunique()})

app_files = files_by_uid(DATA_ROOT / 'app_usage', '.csv')
app_rows = []
for uid, paths in app_files.items():
    try:
        a = pd.concat([app_daily(p) for p in paths])
        a = a.groupby(level=0).max()
        a['uid'] = uid
        app_rows.append(a.reset_index())
    except Exception as e:
        print('Could not process app_usage for', uid, '->', e)

app_feats = pd.concat(app_rows, ignore_index=True)
print('app features:', app_feats.shape, '| participants:', app_feats['uid'].nunique())
display(app_feats.head())

## Cell 6 - EMA answers (reference only) per day
Loads Stress and Sleep EMA and converts each answer's time to the local day.
- Stress: we only **count** answers per level (see assumption 4).
- Sleep `hour` is stored as an answer **code**. I convert code -> hours using the options text in `EMA_definition.json` (never guessed). The option `<3` has no exact number, so it becomes empty (NaN) and is counted below.
- The raw JSON also has a strange key called `null` (it holds location/option text). It is ignored.

In [ ]:
with open(DATA_ROOT / 'EMA' / 'EMA_definition.json') as fh:
    ema_def = json.load(fh)

def options_for(category, question_id):
    for item in ema_def:
        if item.get('name') == category:
            for q in item['questions']:
                if q.get('question_id') == question_id:
                    return q.get('options', '')
    return ''

# "[1]<3, [2]3.5, [3]4, ..." -> {1: nan, 2: 3.5, 3: 4.0, ...}
hour_map = {}
for code, label in re.findall(r'\[(\d+)\]([^,\[]+)', options_for('Sleep', 'hour')):
    label = label.strip()
    hour_map[int(code)] = float(label) if re.fullmatch(r'[\d.]+', label) else np.nan
print('sleep hour code -> hours map:', hour_map)

def load_ema_long(category):
    frames = []
    for f in sorted((ema_resp / category).glob('*.json')):
        with open(f) as fh:
            data = json.load(fh)
        if not data:
            continue
        df = pd.DataFrame(data)
        df['uid'] = uid_from_name(f.name)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out['date'] = to_local(out['resp_time']).dt.date
    return out

stress_long = load_ema_long('Stress')
stress_long['stress_level_code'] = pd.to_numeric(stress_long['level'], errors='coerce')
sleep_long = load_ema_long('Sleep')
for col in ('hour', 'rate', 'social'):
    sleep_long[col] = pd.to_numeric(sleep_long[col], errors='coerce')
sleep_long['sleep_hours_reported'] = sleep_long['hour'].map(hour_map)

print('stress responses:', len(stress_long), '| with a level:', int(stress_long['stress_level_code'].notna().sum()))
print('sleep responses :', len(sleep_long), '| with an hour code:', int(sleep_long['hour'].notna().sum()),
      '| hour code "<3" (no exact value):', int((sleep_long['hour'] == 1).sum()))

# --- daily summaries ---
stress_daily = (stress_long.dropna(subset=['stress_level_code'])
                .assign(stress_level_code=lambda d: d['stress_level_code'].astype(int))
                .pivot_table(index=['uid', 'date'], columns='stress_level_code', values='resp_time',
                             aggfunc='count', fill_value=0))
stress_daily.columns = [f'stress_n_level_{c}' for c in stress_daily.columns]
stress_daily['stress_n_resp'] = stress_daily.sum(axis=1)

sleep_daily = (sleep_long.groupby(['uid', 'date'])
               .agg(sleep_hours_reported=('sleep_hours_reported', 'mean'),
                    sleep_rate_code=('rate', 'mean'),
                    sleep_social_code=('social', 'mean')))
ema_daily = stress_daily.join(sleep_daily, how='outer').reset_index()
print('EMA daily table:', ema_daily.shape)
display(ema_daily.head())

## Cell 7 - Put everything together and save
Joins all feature groups on (participant, day). The rows are the days where at least one **sensing** source has data. EMA columns are attached where they exist.
Saved files (in `OUT_DIR`): `studentlife_daily_features.csv`, `studentlife_ema_stress_long.csv`, `studentlife_ema_sleep_long.csv`.
Participant id is kept only for grouping/splitting later - it must **never** be a model feature.

In [ ]:
key = ['uid', 'date']
daily = interval_feats.merge(activity_feats, on=key, how='outer') \
                      .merge(app_feats, on=key, how='outer') \
                      .merge(ema_daily, on=key, how='left')
daily = daily.sort_values(key).reset_index(drop=True)

# EMA days with no sensing row at all (reported, not silently dropped)
sensed = daily[key].drop_duplicates()
ema_only = ema_daily.merge(sensed, on=key, how='left', indicator=True)
n_ema_only = int((ema_only['_merge'] == 'left_only').sum())
print('daily table:', daily.shape, '| participants:', daily['uid'].nunique())
print('EMA participant-days that have NO sensing row:', n_ema_only, 'of', len(ema_daily))

daily.to_csv(OUT_DIR / 'studentlife_daily_features.csv', index=False)
stress_long.drop(columns=['null', 'location'], errors='ignore').to_csv(OUT_DIR / 'studentlife_ema_stress_long.csv', index=False)
sleep_long.drop(columns=['null', 'location'], errors='ignore').to_csv(OUT_DIR / 'studentlife_ema_sleep_long.csv', index=False)
print('saved to', OUT_DIR)
display(daily.head())

## Cell 8 - Sanity checks (what to read carefully)
1. **Days per participant** - how much data each person really has.
2. **Missing values per column** - which features are often empty.
3. **Sleep proxy vs reported sleep** - a first honest check of the proxy (Spearman correlation, only on days that have both). A weak or zero correlation would mean the proxy is NOT trustworthy yet.
4. **Days with a stress answer** - how many rows could ever have a stress reference label.

In [ ]:
print('--- days per participant ---')
print(daily.groupby('uid').size().describe().round(1))

print('\n--- fraction missing per column ---')
miss = daily.drop(columns=key).isna().mean().sort_values(ascending=False).round(3)
display(miss.to_frame('fraction_missing'))

print('\n--- sleep proxy vs reported sleep (Spearman) ---')
both = daily[['sleep_proxy_hours', 'sleep_hours_reported']].dropna()
print('days with both:', len(both))
if len(both) >= 10:
    print('Spearman correlation:', round(both['sleep_proxy_hours'].corr(both['sleep_hours_reported'], method='spearman'), 3))
display(daily[['sleep_proxy_hours', 'sleep_hours_reported']].describe().round(2))

print('\n--- days with at least one stress answer ---')
print(int((daily['stress_n_resp'].fillna(0) > 0).sum()), 'of', len(daily), 'participant-days')
print('stress answers per level (all days):')
print(daily[[c for c in daily.columns if c.startswith('stress_n_level_')]].sum())

## Cell 9 - What to send back to Claude
Send the printed output of Cells 3, 4, 5, 6, 7 and 8 (text or screenshots). Do not worry about the long tables, the summaries are enough.
Next step after we check them: decide the preliminary target definition and the before-the-answer time windows (leakage control).